In [2]:
"""
Modern UOV (Reformüle Edilmiş Unbalanced Oil and Vinegar) İmza Şeması
==========================================================================

Bu modül, Unbalanced Oil and Vinegar (UOV) imza şemasının "modern"
(reformüle edilmiş) inşa yöntemine dayanan öğretici bir SageMath
simülasyonunu içermektedir.

Modern UOV, Klasik UOV'den Nasıl Farklıdır?
-----------------------------------------------
Klasik UOV inşasında açık anahtar, önce Oil x Oil bloğu SIFIR olan bir
merkezi dönüşüm F rastgele üretilip, ardından gizli bir doğrusal
dönüşüm T ile P(x) = F(T(x)) biçiminde MASKELENEREK elde edilir. Bu
yaklaşımda gizli anahtar hem T dönüşümünü hem de F'yi saklamayı
gerektirir.

MODERN (reformüle edilmiş) yöntemde ise bu iki adım BİRLEŞTİRİLİR:
gizli anahtar doğrudan bir O matrisi (Vinegar uzayından Oil uzayına
bir dönüşüm) olarak seçilir ve açık anahtar matrisleri P_k, bu O
matrisinin çekirdeğini (gizli Oil uzayını) sıfırlayacak biçimde
DOĞRUDAN inşa edilir -- yani P = F o T bileşimi hiç hesaplanmaz, P_k
matrisleri baştan itibaren "O'yu sıfırlayan" özelliğe sahip olacak
şekilde kurulur. Bu, klasik UOV ile MATEMATİKSEL olarak eşdeğerdir
(ikisi de aynı trapdoor yapısını temsil eder) ancak anahtar boyutunu
ve hesaplama karmaşıklığını azaltan, literatürde sıkça kullanılan
(örn. UOV NIST PQC teklifi) bir optimizasyondur.

Gizli Oil Uzayının Cebirsel Temsili
----------------------------------------
Gizli Oil uzayı, O_bar = [O; I_o] (yani üstte v x o boyutunda O matrisi,
altta o x o boyutunda birim matris I_o olan) blok matrisin SÜTUN
UZAYI olarak tanımlanır. Bir vektör bu uzaydaymışçasına
w = O_bar * x (x in F_q^o) biçiminde yazılabilir; ilk v koordinatı
O*x, son o koordinatı ise x'in kendisidir. Modern UOV'nin temel
kısıtlaması şudur: her açık anahtar matrisi P_k için,

    O_bar^T * P_k * O_bar

matrisi ALTERNE (alternating) bir matris olmalıdır -- yani tüm x
vektörleri için x^T (O_bar^T P_k O_bar) x = 0 sağlanmalıdır. Bu,
kuadratik formun gizli Oil uzayı üzerinde ÖZDEŞ OLARAK SIFIR
olduğu (dolayısıyla Oil uzayının kuadratik denklem sisteminin bir
çözüm uzayı olduğu) anlamına gelir ve UOV'nin trapdoor mekanizmasının
matematiksel özüdür. Tek karakteristikte alterne matris ANTİ-SİMETRİK
(skew-symmetric, M^T = -M) matrisle örtüşür; karakteristik 2'de ise
anti-simetriklik simetriklikle çakıştığından (çünkü -1 = 1), alterne
koşulu SIFIR KÖŞEGENLİ SİMETRİK matris olarak ifade edilir -- bu
ayrım _is_alternating_matrix metodunda açıkça ele alınmaktadır.

P_k Matrisinin Blok Blok İnşası
------------------------------------
Her açık anahtar matrisi P_k, üç bloktan inşa edilir:
    P_k = [ P1   P2 ]
          [  0   P3 ]
- P1 (v x v, üst üçgensel): Vinegar x Vinegar terimleri, tamamen
  serbest ve rastgele.
- P2 (v x o): Vinegar x Oil terimleri, tamamen serbest ve rastgele.
- P3 (o x o): Oil x Oil terimleri, RASTGELE DEĞİL, P1 ve P2'den
  DETERMİNİSTİK olarak hesaplanır; amaç, O_bar^T P_k O_bar
  matrisinin alterne olmasını garanti etmektir (aşağıda generate_keys
  içinde ayrıntılı türetilmiştir).
Alt sol blok özdeş olarak SIFIRDIR; bu, P_k'nın üst üçgensel bir
temsilinin doğal bir sonucudur (kuadratik form matrisleri simetrik
kısma bölünmeden tek bir üst üçgensel matrisle temsil edilebilir).

İmzalama: Doğrusallaştırma (Linearization) Yaklaşımı
---------------------------------------------------------
Modern UOV'de imzalama, klasik UOV'deki "merkezi dönüşümü katman
katman tersine çevirme" yerine, DOĞRUDAN açık anahtar matrislerinden
türetilen bir DOĞRUSAL sistemin kurulup çözülmesine dayanır:
Vinegar değişkenleri v rastgele sabitlenip Oil değişkenleri x
bilinmeyen kabul edildiğinde, s = [v + O*x; x] imza formunun
P_k(s) = t_k denklemlerinde yerine konması, x'e göre TAM OLARAK
DOĞRUSAL bir sistem L*x = t - y üretir (bu, kuadratik formun gizli
Oil uzayında sıfırlanması özelliğinin doğrudan bir sonucudur). Bu
sistem çözülebilirse imza elde edilir; çözülemezse (L tekil ise)
yeni bir rastgele Vinegar vektörüyle yeniden denenir.

Bu Simülasyonun Sınırları ve Amaçları
-----------------------------------------
- Bu kod GÜVENLİ parametreler üretmek için değil, Modern UOV'nin
  "P = F o T" yerine "P'yi doğrudan O'yu sıfırlayacak şekilde inşa et"
  yaklaşımının cebirsel işleyişini somut biçimde göstermek amacıyla
  hazırlanmıştır.
- Küçük bir sonlu cisim (örn. GF(5)) ve az sayıda değişken üzerinde
  çalıştırılarak ara adımların (bloklar, kontrol matrisleri, doğrusal
  sistemler) okunabilir biçimde ekrana yazdırılması hedeflenmiştir.
- İmzalama fonksiyonu (sign), her denemede rastgele bir Vinegar vektörü
  seçtiğinden BAŞARISIZ olabilir (L matrisi tekil çıkabilir); bu
  nedenle en fazla 100 deneme yapılan bir RETRY mekanizması içerir --
  bu, Oil-Vinegar ailesi şemalarının DOĞASINDA olan, kriptografik
  güvenliği etkilemeyen normal bir durumdur.

Algoritmanın Genel Akışı
--------------------------
1. Anahtar Üretimi (generate_keys):
   Gizli O matrisi rastgele üretilir; her açık anahtar matrisi P_k,
   P1 ve P2 blokları rastgele seçilip P3 bloğu deterministik olarak
   hesaplanarak (O_bar^T P_k O_bar alterne olacak şekilde) inşa edilir.

2. İmzalama (sign):
   Rastgele bir Vinegar vektörü seçilir; y = P(v||0) sabit terimi ve
   L doğrusal sistem matrisi (P_k'nın x'e göre türevinden gelen S_i
   matrisleri aracılığıyla) kurulur; L*x = t - y çözülerek Oil
   ön-görüntüsü x bulunur ve imza s = [v + O*x; x] olarak oluşturulur.

3. Doğrulama (verify):
   İmza, açık anahtar matrisleri (ve isteğe bağlı olarak polinomları)
   üzerinden değerlendirilip sonucun orijinal mesajla eşleşip
   eşleşmediği kontrol edilir.

Bu implementasyon; yüksek lisans tezi kapsamında Modern UOV'nin
"doğrudan O'yu sıfırlayan P inşası" ve "doğrusallaştırma tabanlı
imzalama" fikirlerini somutlaştırmak amacıyla eğitim/araştırma amaçlı
geliştirilmiştir.


-------------
Bu kod SageMath ortamında çalıştırılmak üzere tasarlanmıştır ve sonlu
cisimler (GF), çok değişkenli polinom halkaları (PolynomialRing), blok
matris inşası (block_matrix), matris cebiri (alt matris çıkarma,
tersinirlik/doğrusal sistem çözümü, transpoze) gibi SageMath'in
yerleşik simgesel/sayısal cebir araçlarına dayanır.



Referans: Tez, Bölüm 6
"""

from sage.all import *

class ModernUOVSignatureScheme:
    """
    Modern (reformüle edilmiş) Unbalanced Oil and Vinegar imza
    şemasının öğretici SageMath simülasyonu.

    Bu sınıf; gizli bir O matrisi (Vinegar uzayından Oil uzayına bir
    doğrusal dönüşüm) üretir ve açık anahtar matrislerini P = F o T
    bileşimini hiç hesaplamadan, doğrudan bu O matrisinin gizli Oil
    uzayını sıfırlayacak biçimde inşa eder; ardından doğrusallaştırma
    tabanlı imzalama ve matris/polinom değerlendirmesiyle doğrulama
    işlemlerini gerçekleştirir.

    Matematiksel Kurulum
    ---------------------
    - F_q = GF(q) : Temel sonlu cisim.
    - R = F_q[x_0, ..., x_{n-1}] : n değişkenli, F_q katsayılı çok
      değişkenli polinom halkası.
    - v : Vinegar değişkenlerinin sayısı (n - m).
    - o : Oil değişkenlerinin sayısı (m, aynı zamanda denklem sayısı).
    - n = v + o : Toplam değişken sayısı.
    - Değişken sıralaması: x_0, ..., x_{v-1} Vinegar; x_v, ..., x_{n-1}
      Oil değişkenleridir.

    Parametreler
    ------------
    q : int
        Temel sonlu cismin eleman sayısı (asal veya asal kuvveti).
    v : int
        Vinegar değişkenlerinin sayısı (n - m).
    o : int
        Oil değişkenlerinin sayısı (m); aynı zamanda üretilecek açık
        anahtar denklemi/matrisi sayısıdır.

    Öznitelikler (Attributes)
    --------------------------
    n : int
        Toplam değişken sayısı (v + o).
    F : FiniteField
        Temel sonlu cisim F_q = GF(q).
    R : MPolynomialRing
        n değişkenli, F_q katsayılı çok değişkenli polinom halkası.
    vars : vector
        R halkasının üreteçlerinden oluşan vektör (x vektörü).
    O_mat : Matrix
        Gizli anahtarı oluşturan v x o boyutunda matris; gizli Oil
        uzayını O_bar = [O_mat; I_o] blok matrisinin sütun uzayı
        olarak tanımlar.
    P_matrices : list of Matrix
        Açık anahtarı oluşturan, her biri n x n boyutunda o adet üst
        üçgensel (P1, P2, 0, P3 bloklarından kurulu) matris.
    P_polys : list of MPolynomial
        Açık anahtar matrislerine karşılık gelen o adet kuadratik
        polinom (P_k(x) = x^T P_k x).
    """
    def __init__(self, q, v, o):
        """
        Modern UOV şemasının parametrelerini, temel sonlu cismi ve
        çok değişkenli polinom halkasını başlatır.

        Parametreler
        ------------
        q : int
            Temel sonlu cismin eleman sayısı.
        v : int
            Vinegar değişkenlerinin sayısı (n - m).
        o : int
            Oil değişkenlerinin sayısı (m).

        Yan Etkiler
        -----------
        self.F, self.R, self.vars öznitelikleri hesaplanır; self.n =
        v + o olarak belirlenir; self.O_mat None, self.P_matrices ve
        self.P_polys boş listeler olarak başlatılır (bunlar ancak
        generate_keys() çağrıldığında doldurulur). Parametre özeti
        ekrana yazdırılır.

        Dönüş Değeri
        ------------
        None
        """
        self.q = q
        self.v = v # Vinegar sayısı (n-m)
        self.o = o # Oil sayısı (m)
        self.n = v + o

        # F_q = GF(q) temel sonlu cismi ve bu cisim üzerinde n
        # değişkenli çok değişkenli polinom halkası R = F_q[x_0,...,x_{n-1}];
        # Modern UOV'nin tüm cebirsel yapısı (açık anahtar matrisleri,
        # kuadratik formlar, doğrulama denklemleri) bu halka üzerinde
        # tanımlanır.
        self.F = GF(q, 'a')
        self.R = PolynomialRing(self.F, self.n, 'x')
        self.vars = vector(self.R, self.R.gens()) # x vektörü

        print("\n" + "="*70)
        print("MODERN UOV (Reformüle edilmiş UOV): DETAYLI ANAHTAR ÜRETİMİ VE İMZALAMA ANALİZİ")
        print(f"Parametreler: GF({q}), v={v}, o={o} (n={self.n})")
        print(f"Değişkenler: Vinegar [x0..x{v-1}], Oil [x{v}..x{self.n-1}]")
        print("="*70)

        self.O_mat = None
        self.P_matrices = []
        self.P_polys = []



    def _is_alternating_matrix(self, M):
        """
        x^T M x = 0 koşulunu tüm x vektörleri için sağlayan matrisleri kontrol eder.
        Tek karakteristikte bu anti-simetrik/skew-symmetric yapı,
        karakteristik 2'de ise sıfır köşegenli simetrik yapı olarak görünür.

        Teorik Gerekçe
        ---------------
        Bir M matrisinin ALTERNE (alternating) olması, ilişkili
        kuadratik formun x^T M x özdeş olarak sıfır olması demektir.
        Tek karakteristikli cisimlerde (char != 2) bu koşul cebirsel
        olarak M^T = -M (anti-simetriklik) ile denktir, çünkü
        x^T M x = 0 açılımı köşegen terimlerin 2*M[i,i] = 0, yani
        M[i,i] = 0 olmasını ve simetrik terimlerin M[i,j] = -M[j,i]
        olmasını zorunlu kılar. KARAKTERİSTİK 2'de ise -1 = 1
        olduğundan anti-simetriklik ile simetriklik cebirsel olarak
        ÇAKIŞIR (M^T = -M ile M^T = M aynı koşuldur); bu durumda
        alterne olma koşulu ayrıca SIFIR KÖŞEGEN şartının da
        (M[i,i] = 0) açıkça denetlenmesini gerektirir -- aksi halde
        salt simetrik bir matris (köşegeni sıfır olmayan) yanlışlıkla
        alterne sayılabilir. Bu metot, bu iki karakteristik durumunu
        ayrı ayrı ele alarak generate_keys() içindeki [KONTROL]
        adımının doğruluğunu güvence altına alır.

        Parametreler
        ------------
        M : Matrix
            Alterne olup olmadığı test edilecek, self.F üzerinde
            tanımlı kare matris (tipik olarak O_bar^T P_k O_bar).

        Dönüş Değeri
        ------------
        bool
            M matrisi (cismin karakteristiğine uygun tanımla) alterne
            ise True; aksi halde False.
        """
        if self.F.characteristic() == 2:
            return M.is_symmetric() and all(M[i, i] == 0 for i in range(M.nrows()))
        else:
            return M.is_skew_symmetric()




    def _check_keys_generated(self):
        """
        Modern UOV açık anahtarının ve gizli O matrisinin üretilip üretilmediğini kontrol eder.

        Bu iç yardımcı metot, sign() ve verify() metotlarının
        generate_keys() çağrılmadan kullanılmasını engelleyen bir
        koruma (guard) katmanıdır; O_mat, P_matrices veya P_polys
        bileşenlerinden herhangi biri eksikse anlamlı bir hata
        mesajıyla erken durdurma sağlar.

        Ön Koşullar
        ------------
        Yoktur (bu metot herhangi bir zamanda çağrılabilir; amacı tam
        olarak diğer metotların ön koşulunu denetlemektir).

        Dönüş Değeri
        ------------
        None
            Anahtarlar eksiksizse sessizce döner.

        Fırlatılan İstisnalar
        -----------------------
        RuntimeError
            self.O_mat None ise, ya da self.P_matrices veya
            self.P_polys boşsa fırlatılır.
        """
        if self.O_mat is None or len(self.P_matrices) == 0 or len(self.P_polys) == 0:
            raise RuntimeError("Önce generate_keys() çağrılmalıdır.")



    def generate_keys(self):
        """
        Modern UOV şemasının gizli anahtarını (O matrisi) rastgele
        üretir ve bundan, açık anahtarı (P_matrices, P_polys) klasik
        "P = F o T" bileşimine hiç başvurmadan DOĞRUDAN inşa eder.

        Algoritmanın Adımları
        -----------------------
        1. Gizli O Matrisi:
           v x o boyutunda rastgele bir O_mat matrisi seçilir. Bu
           matris, gizli Oil uzayını O_bar = [O_mat; I_o] blok
           matrisinin (v+o) x o boyutundaki SÜTUN UZAYI olarak
           tanımlar; yani gizli Oil uzayındaki her vektör
           O_bar * x (x in F_q^o) biçiminde yazılabilir.

        2. Her k = 0, ..., o-1 için Açık Anahtar Matrisi P_k İnşası:
           a) P1 (v x v, üst üçgensel): Vinegar x Vinegar terimleri,
              tamamen serbestçe rastgele seçilir.
           b) P2 (v x o): Vinegar x Oil terimleri, tamamen serbestçe
              rastgele seçilir.
           c) P3 (o x o) -- DETERMİNİSTİK HESAPLAMA: Amaç,
              O_bar^T * P_k * O_bar matrisinin ALTERNE olmasını
              (yani gizli Oil uzayında kuadratik formun özdeş olarak
              sıfırlanmasını) garanti etmektir. Ara büyüklük
                  Q = O_mat^T * P1 * O_mat + O_mat^T * P2
              hesaplanır (bu, P_k'nın Oil uzayı üzerindeki kuadratik
              formunun P1 ve P2'den gelen katkısını temsil eder).
              P3 üst üçgensel olarak seçilir ve (Q + P3) toplamının
              alterne olması istenen köşegen ile üst üçgen şu iki
              denklemi üretir:
                  Köşegen  (r=c):  (Q+P3)[r,r] = 0  =>  P3[r,r] = -Q[r,r]
                  Üst üçgen (r<c): (Q+P3)[r,c] = -(Q+P3)[c,r]
                                   =>  P3[r,c] = -Q[c,r] - Q[r,c]
              Bu iki bağıntı doğrudan kod içinde uygulanarak P3
              tamamen deterministik biçimde (rastgelelik OLMADAN)
              hesaplanır.
           d) Tam P Matrisi: Bloklar
                  P_k = [ P1   P2 ]
                        [  0   P3 ]
              biçiminde birleştirilir (alt sol blok özdeş sıfırdır).
           e) Polinom Hali: poly = x^T * P_k * x hesaplanarak P_k'ya
              karşılık gelen kuadratik polinom elde edilir.
           f) Sağlama (Doğrulama Adımı): Check_Mat =
              O_bar^T * P_k * O_bar hesaplanır ve
              _is_alternating_matrix ile bu matrisin gerçekten
              alterne olduğu (yani P3'ün doğru inşa edildiği)
              denetlenir; sonuç ekrana [ONAY]/[HATA] olarak yazdırılır.

        Yan Etkiler
        -----------
        self.O_mat, self.P_matrices ve self.P_polys öznitelikleri bu
        metot tarafından (yeniden) doldurulur. Metot birden fazla kez
        çağrılırsa önceki anahtar bileşenleri temizlenip yeniden
        üretilir. Süreç boyunca ayrıntılı ara çıktılar (bloklar,
        tam P matrisleri, polinomlar, alternelik kontrolü) ekrana
        yazdırılır.

        Dönüş Değeri
        ------------
        None
        """
        # generate_keys() birden fazla kez çağrılırsa eski anahtar bileşenleri temizlenir.
        self.P_matrices = []
        self.P_polys = []
        self.O_mat = None

        print("\n" + "*"*30 + " ADIM 1: ANAHTAR İNŞASI (MODERN YÖNTEM) " + "*"*30)
        print("Mantık: P = F o T yapmak yerine, O matrisini (Gizli Uzay) sıfırlayan P matrisleri doğrudan inşa edilir.")

        # 1. GİZLİ MATRİS O
        # O_mat, gizli Oil uzayını tanımlayan temel yapı taşıdır;
        # Vinegar koordinatlarından Oil koordinatlarına giden bu
        # doğrusal dönüşüm sayesinde gizli uzay O_bar = [O_mat; I_o]
        # matrisinin sütun uzayı olarak temsil edilir.
        self.O_mat = random_matrix(self.F, self.v, self.o)

        print(f"\n[1.1] Gizli O Matrisi (Vinegar x Oil) - ({self.v}x{self.o}):")
        print(self.O_mat)

        # Arka Kapı Matrisi (O_bar) = [O / I]
        # O_bar, gizli Oil uzayının (v+o) boyutlu ortam uzayındaki
        # temsilini sağlayan blok matristir; her gizli Oil uzayı
        # vektörü O_bar * x (x in F_q^o) biçiminde parametrize edilir.
        I_m = identity_matrix(self.F, self.o)
        O_bar = block_matrix([[self.O_mat], [I_m]])

        # 2. AÇIK ANAHTAR BİLEŞENLERİ
        print(f"\n[1.2] Açık Anahtar Matrislerinin (P_k) Blok Blok İnşası")

        for k in range(self.o):
            print(f"\n   >>> Polinom P_{k} İnşası <<<")

            # A) P1 (v x v) - Üst Üçgen
            # Vinegar x Vinegar terimleri; Modern UOV'de bu blok
            # üzerinde hiçbir kısıtlama yoktur, tamamen serbestçe
            # rastgele seçilebilir (Oil uzayının sıfırlanma koşuluna
            # bu blok yalnızca P3 üzerinden dolaylı katkıda bulunur).
            P1 = matrix(self.F, self.v, self.v)
            for i in range(self.v):
                for j in range(i, self.v):
                    P1[i, j] = self.F.random_element()

            # B) P2 (v x o) - Rastgele
            P2 = random_matrix(self.F, self.v, self.o)

            print(f"   [Blok 1] P1 (Vinegar x Vinegar):\n{P1}")
            print(f"   [Blok 2] P2 (Vinegar x Oil):\n{P2}")

            # C) P3 (o x o) - HESAPLAMA
            # Kural: O^T * P1 * O + O^T * P2 + P3  -> Anti-Simetrik Olmalı
            # Q, P1 ve P2'nin gizli Oil uzayı üzerindeki kuadratik
            # forma yaptığı katkıyı temsil eder; P3 bu katkıyı
            # "dengeleyerek" toplamın alterne olmasını sağlayacak
            # şekilde geriye doğru hesaplanır.
            Q = self.O_mat.transpose() * P1 * self.O_mat + self.O_mat.transpose() * P2

            # C) P3 (o x o) - Deterministik hesaplama
            # Amaç: Q + P3 anti-simetrik olsun.
            # Burada Q = O^T P1 O + O^T P2.
            #
            # P3 üst üçgensel seçilir.
            # Köşegen için:       (Q + P3)_{ii} = 0  => P3_{ii} = -Q_{ii}
            # Üst üçgen için:     (Q + P3)_{rc} = -(Q + P3)_{cr}
            #                    Q_{rc} + P3_{rc} = -Q_{cr}
            #                    P3_{rc} = -Q_{cr} - Q_{rc}

            P3 = matrix(self.F, self.o, self.o)
            for r in range(self.o):
                for c in range(self.o):
                    if r == c:
                        P3[r, c] = -Q[r, c]
                    elif r < c:
                        P3[r, c] = -Q[c, r] - Q[r, c]

            print(f"   [Blok 3] P3 (Oil x Oil) - Deterministik Olarak Hesaplanmıştır:\n{P3}")

            # D) Tam P Matrisi (Açık Anahtar Matrisi)
            # P = [ P1  P2 ]
            #     [ 0   P3 ]
            # Alt sol blok özdeş sıfırdır; bu, P_k'nın üst üçgensel
            # bir kuadratik form temsili olmasının doğal bir sonucudur.
            Zero_block = matrix(self.F, self.o, self.v)
            P_full = block_matrix([[P1, P2], [Zero_block, P3]])

            self.P_matrices.append(P_full)

            print(f"   [SONUÇ] Tam P_{k} Matrisi (Açık Anahtar Matrisi):\n{P_full}")

            # E) Polinom Hali
            poly = self.vars * P_full * self.vars # x^T P x
            self.P_polys.append(poly)

            print(f"   [SONUÇ] P_{k}(x) Polinomu:\n   {poly}")

            # F) SAĞLAMA
            # Bu denetim, P_k inşasının doğruluğunu gözle teyit etmek
            # amacıyla eklenmiştir: O_bar^T P_k O_bar matrisinin
            # gerçekten alterne (dolayısıyla gizli Oil uzayında
            # kuadratik formun özdeş olarak sıfır) olduğu doğrulanır.
            Check_Mat = O_bar.transpose() * P_full * O_bar
            is_valid = self._is_alternating_matrix(Check_Mat)

            print(f"   [KONTROL] O_bar^T P_{k} O_bar Matrisi:\n{Check_Mat}")
            print(f"   [KONTROL] O Uzayında Sıfırlanma: {'BAŞARILI' if is_valid else 'HATA'}")

    def sign(self, message_vec):
        """
        Verilen bir mesaj vektörü için, gizli O matrisi ve açık
        anahtar matrislerinden türetilen bir DOĞRUSAL sistem
        aracılığıyla Modern UOV imzasını üretir.

        Algoritmanın Adımları
        -----------------------
        1. Rastgele Vinegar Seçimi:
           v boyutunda rastgele bir vinegar vektörü v_vec seçilir.

        2. Sabit Terim y'nin Hesaplanması:
           v_vec, Oil koordinatları sıfırla doldurularak
           v_padded = (v_vec || 0) elde edilir ve her P_k matrisi için
           y_k = v_padded^T * P_k * v_padded hesaplanarak y vektörü
           oluşturulur. Bu, P_k(v||x) ifadesinin x=0 noktasındaki
           (yani yalnızca Vinegar katkısından gelen) sabit kısmını
           temsil eder.

        3. Doğrusal Sistem Matrisi L'nin Kurulması:
           Her P_k matrisinden P1 (v x v) ve P2 (v x o) alt blokları
           çıkarılır; S_i = (P1 + P1^T) * O_mat + P2 hesaplanır (bu,
           P_k(v || x) ifadesinin x'e göre TÜREVİNİN v_vec ile
           çarpımına karşılık gelen, doğrusal katkıyı temsil eden
           matristir -- P1 tek taraflı olduğundan türev alınırken
           P1 + P1^T simetrikleştirmesi ortaya çıkar). L matrisinin
           k'ıncı satırı v_vec * S_i olarak elde edilir.

        4. Doğrusal Sistemin Çözümü:
           L * x = t - y denklemi kurulur (t = message_vec, y = 2.
           adımda hesaplanan sabit terim) ve Sage'in solve_right
           metoduyla çözülür. Çözüm bulunursa x (Oil ön-görüntüsü)
           elde edilir; L tekil ise (çözüm yoksa), yeni bir rastgele
           Vinegar vektörüyle yeniden denenir (RETRY mekanizması; bu,
           Oil-Vinegar ailesi şemalarında beklenen, güvenliği
           etkilemeyen bir durumdur).

        5. İmza Vektörünün Oluşturulması:
           s = [v_vec + O_mat * x ; x] biçiminde birleştirilerek
           nihai imza elde edilir (üst kısım gizli Oil uzayına
           O_bar aracılığıyla geri taşınmış "kirletilmiş" Vinegar
           bileşenini, alt kısım ise doğrudan çözülen Oil bileşenini
           temsil eder).

        Ön Koşullar
        ------------
        generate_keys() metodunun bu metottan önce çağrılmış olması
        gerekir (_check_keys_generated aracılığıyla denetlenir).
        message_vec'in uzunluğu self.o'ya eşit olmalıdır.

        Parametreler
        ------------
        message_vec : vector
            Uzunluğu self.o (=m) olan, F_q üzerinde tanımlı, imzalanacak
            (tipik olarak bir hash fonksiyonundan gelen) mesaj vektörü.

        Dönüş Değeri
        ------------
        vector veya None
            Başarılı olursa, uzunluğu self.n olan geçerli bir Modern
            UOV imzası; en fazla 100 denemede hiçbir Vinegar vektörü
            L'yi tersinir kılmazsa None.

        Fırlatılan İstisnalar
        -----------------------
        RuntimeError
            Anahtarlar henüz üretilmemişse (_check_keys_generated
            aracılığıyla).
        ValueError
            message_vec'in uzunluğu self.o'ya eşit değilse.
        """
        self._check_keys_generated()

        if len(message_vec) != self.o:
            raise ValueError(f"Mesaj vektörünün uzunluğu o=m={self.o} olmalıdır.")

        print("\n" + "*"*30 + " ADIM 2: İMZALAMA (LINEERLEŞTİRME) " + "*"*30)
        print(f"   İmzalanacak Mesaj (t): {message_vec}")

        attempts = 0
        while attempts < 100:
            attempts += 1
            print(f"\n   --- Deneme {attempts} ---")

            # 1. Vinegar Seçimi
            # Bu vektör, doğrusal sistemin katsayılarını (L) ve sabit
            # terimini (y) belirleyen serbest parametredir; Modern
            # UOV'de her deneme için yeniden rastgele seçilir.
            v_vec = random_vector(self.F, self.v)
            print(f"   1. Rastgele Vinegar (v): {v_vec}")

            # 2. y Vektörü (Sabit Kısım)
            # y_k = P_k(v || 0), yani Oil değişkenleri sıfırlanmış
            # haldeyken P_k'nın aldığı değer; bu, L*x = t - y
            # denkleminin sabit (x'ten bağımsız) kısmını oluşturur.
            v_padded = vector(self.F, list(v_vec) + [0]*self.o)
            y_vec_list = []
            for P_mat in self.P_matrices:
                val = v_padded * P_mat * v_padded
                y_vec_list.append(val)
            y_vec = vector(self.F, y_vec_list)

            print(f"   2. y = P(v||0) Değeri: {y_vec}")

            # 3. L Matrisi (Lineer Sistem)
            # L_ij = v^T * S_i  where S_i = (P1 + P1^T)O + P2

            L_rows = []
            print(f"   3. L Matrisi Kuruluyor...")

            for k in range(self.o):
                P_full = self.P_matrices[k]
                P1 = P_full.submatrix(0, 0, self.v, self.v)
                P2 = P_full.submatrix(0, self.v, self.v, self.o)

                # Simetrikleşmiş P1 (Dikkat: Sage'de x^T P x formunda P1 tek taraflıdır,
                # Türev alırken P1 + P1^T gelir)
                # S_i, P_k(v||x) ifadesinin x'e göre doğrusal katkısını
                # taşıyan matristir; v_vec ile çarpıldığında L'nin
                # k'ıncı satırı elde edilir.
                S_i = (P1 + P1.transpose()) * self.O_mat + P2

                row = v_vec * S_i
                L_rows.append(row)

            L = matrix(self.F, L_rows)
            print(f"      L Matrisi ({self.o}x{self.o}):\n{L}")

            # 4. Denklem: L * x = t - y
            rhs = message_vec - y_vec
            print(f"      Sağ Taraf (t - y): {rhs}")
            print("      Açık Lineer Denklem Sistemi:")

            x_names = [f"oil_{j}" for j in range(self.o)]

            for row_idx in range(L.nrows()):
                terms = []
                for col_idx in range(L.ncols()):
                    coeff = L[row_idx, col_idx]
                    if coeff != 0:
                        terms.append(f"({coeff})*{x_names[col_idx]}")

                lhs = " + ".join(terms) if terms else "0"
                print(f"         Denklem {row_idx+1}: {lhs} = {rhs[row_idx]}")

            try:
                x_sol = L.solve_right(rhs)
                print(f"   4. [BAŞARILI] x Çözümü (Oil Ön-Görüntüsü): {x_sol}")

                # 5. İmza Oluşumu
                # s = [v + O*x]
                #     [    x  ]
                # Üst kısım, çözülen Oil bileşeninin O_mat aracılığıyla
                # Vinegar koordinatlarına "geri yansıtılmış" katkısıyla
                # birleştirilmiş halidir; bu, imzanın gerçekten gizli
                # Oil uzayının O_bar parametrizasyonuna uyduğunu
                # garanti eder.
                top_part = v_vec + self.O_mat * x_sol
                bottom_part = x_sol
                signature = vector(self.F, list(top_part) + list(bottom_part))

                print(f"   5. İmza (s) Hesaplanıyor (s = [v + O*x, x]):")
                print(f"      Üst Kısım (v + O*x): {top_part}")
                print(f"      Alt Kısım (x): {bottom_part}")
                print(f"      Final İmza: {signature}")
                return signature

            except ValueError:
                # L matrisi tekil (tersinir olmayan doğrusal sistem)
                # ise bu Vinegar seçimi işe yaramaz; bu durumda tüm
                # süreç yeni bir rastgele Vinegar vektörüyle baştan
                # denenmelidir.
                print("   [HATA] L*x = t-y lineer sistemi çözülemedi. Yeni vinegar seçiliyor...")

        return None

    def verify(self, message, signature, show_polynomial_check=False):
        """
        Verilen bir imzanın, açık anahtar matrisleri (ve isteğe bağlı
        olarak açık anahtar polinomları) aracılığıyla verilen mesaja
        karşılık gelip gelmediğini doğrular.

        Teorik Gerekçe
        ---------------
        Modern UOV (ve genel olarak çok değişkenli kuadratik imza
        şemaları) tek yönlü bir kapı fonksiyonuna dayanır: P(x)'i
        (gizli anahtar bilgisi olmadan) HERHANGİ bir hedef için
        tersine çevirmek hesaplama açısından zordur, ancak P(x)'i
        belirli bir x noktasında DEĞERLENDİRMEK (ileri yön) kolaydır.
        Doğrulama, tam olarak bu kolay yönü kullanır: imza doğrudan
        her P_k matrisinde s^T P_k s biçiminde değerlendirilir ve
        sonucun mesajla eşleşip eşleşmediği kontrol edilir; bu adım
        hiçbir gizli anahtar bilgisi gerektirmez. show_polynomial_check
        seçeneği etkinleştirildiğinde, aynı değerlendirme AYRICA açık
        anahtar polinomu (P_polys) üzerinden de tekrarlanır; bu,
        matris temsili x^T P x ile polinom temsilinin (P_polys içinde
        saklanan) tutarlı olduğunu gösteren bir çapraz kontrol
        (sanity check) işlevi görür.

        Ön Koşullar
        ------------
        generate_keys() metodunun bu metottan önce çağrılmış olması
        gerekir (_check_keys_generated aracılığıyla denetlenir).
        message'ın uzunluğu self.o'ya, signature'ın uzunluğu self.n'ye
        eşit olmalıdır.

        Parametreler
        ------------
        message : vector
            Uzunluğu self.o (=m) olan, F_q üzerinde tanımlı orijinal
            mesaj vektörü.
        signature : vector
            Uzunluğu self.n olan, F_q üzerinde tanımlı, doğrulanacak
            imza vektörü (tipik olarak sign() metodunun çıktısı).
        show_polynomial_check : bool, optional
            True ise, her P_k için matris değerlendirmesine (s^T P_k s)
            ek olarak polinom değerlendirmesi (P_polys[k] içine
            imzanın yerleştirilmesi) de yapılır ve iki sonucun
            tutarlılığı ekrana yazdırılır. Varsayılan: False.

        Dönüş Değeri
        ------------
        bool
            İmza, tüm P_k matrislerinde değerlendirildiğinde mesajla
            tam olarak eşleşiyorsa True; aksi halde False.

        Fırlatılan İstisnalar
        -----------------------
        RuntimeError
            Anahtarlar henüz üretilmemişse (_check_keys_generated
            aracılığıyla).
        ValueError
            message'ın uzunluğu self.o'ya veya signature'ın uzunluğu
            self.n'ye eşit değilse.
        """
        self._check_keys_generated()

        if len(message) != self.o:
            raise ValueError(f"Mesaj vektörünün uzunluğu o=m={self.o} olmalıdır.")

        if len(signature) != self.n:
            raise ValueError(f"İmza vektörünün uzunluğu n={self.n} olmalıdır.")

        print("\n" + "*"*30 + " ADIM 3: DOĞRULAMA " + "*"*30)

        res_list = []
        val_dict = {self.vars[i]: signature[i] for i in range(self.n)}

        for idx, P_mat in enumerate(self.P_matrices):
            # Matris çarpımı ile doğrulama: s^T P_k s
            val_matrix = signature * P_mat * signature
            res_list.append(val_matrix)

            if show_polynomial_check:
                # Aynı açık anahtar polinomu doğrudan imza üzerinde değerlendirilir.
                val_poly = self.P_polys[idx].subs(val_dict)

                print(f"   P_{idx}(s):")
                print(f"      Matris değerlendirmesi : {val_matrix}")
                print(f"      Polinom değerlendirmesi: {val_poly}")

                if val_matrix == val_poly:
                    print("      [ONAY] Matris ve polinom değerlendirmeleri aynı.")
                else:
                    print("      [UYARI] Matris ve polinom değerlendirmeleri farklı!")

        check_vec = vector(self.F, res_list)

        print(f"\n   Hesaplanan P(s): {check_vec}")
        print(f"   Orijinal Mesaj : {message}")

        return check_vec == message



In [4]:
# ============================================================================
# SENARYO: MODERN UOV İMZA ŞEMASI İÇİN ÖĞRETİCİ SİMÜLASYON
# ============================================================================
# Bu örnekte GF(5) üzerinde küçük parametreli bir Modern UOV sistemi kurulur.
#
# Parametreler:
#   q = 5  : Sonlu cisim GF(5)
#   v = 4  : Vinegar değişkenlerinin sayısı, yani n-m
#   o = 2  : Oil değişkenlerinin sayısı, yani m
#   n = 6  : Toplam değişken sayısı
#
# Gizli anahtar:
#   O ∈ GF(q)^{v x o}
#
# Açık anahtar:
#   P = (P_0, ..., P_{o-1})
#
# İmzalama:
#   Rastgele v ∈ GF(q)^v seçilir.
#   L*x = t-y lineer sistemi çözülür.
#   İmza s = [v + O*x || x] olarak hesaplanır.
# ============================================================================

print("\n\n" + "*"*60)
print(">>> MODERN UOV SİMÜLASYONU <<<")
print("*"*60)

my_q = 5
my_v = 4
my_o = 2

uov = ModernUOVSignatureScheme(q=my_q, v=my_v, o=my_o)
uov.generate_keys()

msg = random_vector(uov.F, my_o)
sig = uov.sign(msg)

if sig is not None:
    if uov.verify(msg, sig):
        print("\nSONUÇ: İMZA GEÇERLİ.")
    else:
        print("\nSONUÇ: GEÇERSİZ.")
else:
    print("\nSONUÇ: İmza üretilemedi.")




************************************************************
>>> MODERN UOV SİMÜLASYONU <<<
************************************************************

MODERN UOV (Reformüle edilmiş UOV): DETAYLI ANAHTAR ÜRETİMİ VE İMZALAMA ANALİZİ
Parametreler: GF(5), v=4, o=2 (n=6)
Değişkenler: Vinegar [x0..x3], Oil [x4..x5]

****************************** ADIM 1: ANAHTAR İNŞASI (MODERN YÖNTEM) ******************************
Mantık: P = F o T yapmak yerine, O matrisini (Gizli Uzay) sıfırlayan P matrisleri doğrudan inşa edilir.

[1.1] Gizli O Matrisi (Vinegar x Oil) - (4x2):
[4 2]
[3 2]
[1 3]
[2 0]

[1.2] Açık Anahtar Matrislerinin (P_k) Blok Blok İnşası

   >>> Polinom P_0 İnşası <<<
   [Blok 1] P1 (Vinegar x Vinegar):
[3 2 4 4]
[0 4 4 2]
[0 0 1 4]
[0 0 0 0]
   [Blok 2] P2 (Vinegar x Oil):
[0 1]
[3 3]
[4 4]
[1 2]
   [Blok 3] P3 (Oil x Oil) - Deterministik Olarak Hesaplanmıştır:
[1 3]
[0 2]
   [SONUÇ] Tam P_0 Matrisi (Açık Anahtar Matrisi):
[3 2 4 4|0 1]
[0 4 4 2|3 3]
[0 0 1 4|4 4]
[0 0 0 0|1 2]
[